# 12. L2any L2both diffloop

Part of the **[Fig. 1 chapter](fig1.md)** — see it for the panel-by-panel map. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)). *Outputs shown are the author's original run.*


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)

In [42]:
import os
import cooler
import pathlib
import numpy as np
import pandas as pd
from scipy.sparse import load_npz, save_npz, vstack, csr_matrix, triu
from scipy.stats import f, zscore, ranksums
from schicluster.cool.utilities import get_chrom_offsets
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor, as_completed
import warnings
warnings.filterwarnings("ignore")


In [92]:
indir = '/data/ENTEx/'
outdir = '/mnt/disks/jupyter/home/jzhou_salk_edu/sky_workdir/ENTEx/analysis/'


In [5]:
meta = pd.read_csv(f'{indir}clustering/merged/group_meta.tsv', header=0, index_col=0, sep='\t')
meta


,L2_any,L2_mc,L2_3c,L2_both,L1,L1_annot,count
group,,,,,,,
c5-c0-PT-1LVAN,c5-c0,c5-c1,c5-c0,c5-c1,c5,Mus Skl,772
c1-c0-4809,c1-c0,c1-c0,c1-c0,c1-c0,c1,Hema Tmem,771
c5-c0-PT-1K2DA,c5-c0,c5-c1,c5-c0,c5-c1,c5,Mus Skl,627
c4-c0-PT-1LGRB,c4-c0,c4-c0,c4-c0,c4-c0,c4,Epi Gas,593
c4-c0-PT-1LVAN,c4-c0,c4-c0,c4-c0,c4-c0,c4,Epi Gas,533
...,...,...,...,...,...,...,...
c8-c9-Other,c8-c9,c8-c0,c8-c4,c8-c0,c8,Epi Ent,1
c8-c0-Other,c8-c0,c8-c0,c8-c0,c8-c0,c8,Epi Ent,1
c15-c1-Other,c15-c1,c15-c0,c15-c0,c15-c0,c15,Hema Tnaive,1


In [6]:
# dmr_group = {}
# for l1,l1_df in meta.groupby('L1_annot'):
#     count = len(l1_df['L2_both'].unique())
#     if count>1:
#         dmr_group[l1] = {}
#         for l2b,l2b_df in l1_df.groupby('L2_both'):
#             tmp = list(l2b_df['L2_any'].unique())
#             if len(tmp)>1:
#                 dmr_group[l1][l2b.replace('-c','-b')] = tmp


In [35]:
dmr_group = {}
for l1,l1_df in meta.groupby('L1_annot'):
    count = len(l1_df['L2_both'].unique())
    if count>1:
        for l2b,l2b_df in l1_df.groupby('L2_both'):
            tmp = list(l2b_df['L2_any'].unique())
            if len(tmp)>1:
                dmr_group[l2b.replace('-c','-b')] = tmp


In [88]:
print(len(dmr_group))

61


In [87]:
import numpy as np
import pandas as pd
from glob import glob
from gliderport.preset import notebook_snakemake


In [91]:
group_list = list(dmr_group.keys())
print(len(group_list))


61


In [94]:
notebook_snakemake(
    work_dir=f'{outdir}L2any_L2both_diffloop/',
    notebook_dir=f'{outdir}L2any_L2both_diffloop/template/',
    groups=group_list,
    default_cpu=3,
    default_mem_gb=12,
    redo_prepare=True,
)

In [ ]:
!snakemake --snakefile Snakefile -j --keep-going

In [ ]:
group_name = 'c10-b0'


In [45]:
group_list = dmr_group[group_name]
ctgroup = [[xx] for xx in group_list]
ctgroup

[['c10-c0'], ['c10-c4']]

In [55]:
count = meta.groupby('L2_any')['count'].sum()
ncell = count.loc[group_list].sum()
print(ncell)


588


In [49]:
coolpath = pd.read_csv(f'{outdir}coollist_L2any.txt', index_col=None, header=None)[0]
coolpath = pd.Series({xx.split('/')[-1].split('.')[0]:'/'.join(xx.split('/')[:-1]) for xx in coolpath.values})
coolpath

c0-c0         /entex/loop/092924-L2any/0/0/c0-c0/c0-c0/c0-c0
c0-c1         /entex/loop/092924-L2any/0/0/c0-c1/c0-c1/c0-c1
c0-c10     /entex/loop/092924-L2any/0/0/c0-c10/c0-c10/c0-c10
c0-c11     /entex/loop/092924-L2any/0/0/c0-c11/c0-c11/c0-c11
c0-c12     /entex/loop/092924-L2any/0/0/c0-c12/c0-c12/c0-c12
                                 ...                        
c10-c15    /entex/loop/092924-L2any/9/9/c10-c15/c10-c15/c...
c10-c16    /entex/loop/092924-L2any/9/9/c10-c16/c10-c16/c...
c10-c17    /entex/loop/092924-L2any/9/9/c10-c17/c10-c17/c...
c10-c2     /entex/loop/092924-L2any/9/9/c10-c2/c10-c2/c10-c2
c10-c3     /entex/loop/092924-L2any/9/9/c10-c3/c10-c3/c10-c3
Length: 420, dtype: object

In [31]:
res = 10000
chrom_size_path = '/ref/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]
bins_df = cooler.binnify(chrom_sizes, res)
chrom_offset = get_chrom_offsets(bins_df)


In [34]:
bkl = pd.read_csv('/ref/blacklist/hg38_bismark_loop_blacklist.bed', sep='\t', header=None, index_col=None)

In [40]:
def compute_anova(c, matrix):
    # c, matrix = args
    ngene = int(chrom_sizes.loc[c] // res) + 1
    bkl_tmp = bkl.loc[(bkl[0]==c), [1,2]].values // res
    cov = np.zeros(ngene)
    for xx,yy in bkl_tmp:
        cov[xx-7:yy+7] = 1
    tot, last = 0, 0
    Esum, E2sum, Elast, E2last, ss_intra = [csr_matrix((ngene, ngene)) for i in range(5)]
    for ctlist in ctgroup:
        for ct in ctlist:
            cool_e = cooler.Cooler(f'{coolpath[ct]}/{ct}.{matrix}.cool')
            E = triu(cool_e.matrix(balance=False, sparse=True).fetch(c))
            cool_e2 = cooler.Cooler(f'{coolpath[ct]}/{ct}.{matrix}2.cool')
            E2 = triu(cool_e2.matrix(balance=False, sparse=True).fetch(c))
            n = cool_e.info['group_n_cells']
            Esum += E * n
            E2sum += E2 * n
            tot += n
            # print(c, ct)
        Egroup = Esum - Elast
        E2group = E2sum - E2last
        Egroup.data = Egroup.data ** 2 / (tot - last)
        ss_intra += (E2group - Egroup)
        Elast = Esum.copy()
        E2last = E2sum.copy()
        last = tot
    Esum.data = Esum.data ** 2 / tot
    ss_total = E2sum - Esum
    ss_intra.data = 1 / ss_intra.data
    ss_total = ss_total.multiply(ss_intra)
    # print(c, ss_total.data.min(), ss_intra.data.min())

    ss_total.data = (ss_total.data - 1) * (tot - len(ctgroup)) / (len(ctgroup) - 1)
    ss_total = ss_total.tocoo()
    bklfilter = np.logical_and(cov[ss_total.row]==0, cov[ss_total.col]==0)
    distfilter = np.logical_and((ss_total.col-ss_total.row)>5, (ss_total.col-ss_total.row)<500)
    idxfilter = np.logical_and(bklfilter, distfilter)
    # print(idxfilter.sum(), len(idxfilter))
    ss_total = csr_matrix((ss_total.data[idxfilter], (ss_total.row[idxfilter], ss_total.col[idxfilter])), (ngene, ngene))
    save_npz(f'{outdir}L2any_L2both_diffloop/{group_name}/{matrix}pv_{c}.npz', ss_total)

    return [c, matrix, tot]



In [50]:
cpu = 3
with ProcessPoolExecutor(cpu) as executor:
    futures = []
    for x in chrom_sizes.index:
        for y in ['Q', 'E', 'T']:
            future = executor.submit(
                compute_anova,
                c=x,
                matrix=y,
            )
            futures.append(future)

    # result = []
    for future in as_completed(futures):
        # result.append(future.result())
        # c1, c2 = result[-1][0], result[-1][1]
        tmp = future.result()
        print(f'{tmp[0]} {tmp[1]} finished')
        

chr1 E finished
chr1 Q finished
chr1 T finished
chr2 Q finished
chr2 E finished
chr2 T finished
chr3 E finished
chr3 Q finished
chr3 T finished
chr4 Q finished
chr4 E finished
chr4 T finished
chr5 Q finished
chr5 E finished
chr6 E finished
chr6 Q finished
chr5 T finished
chr7 E finished
chr7 Q finished
chr6 T finished
chr8 Q finished
chr7 T finished
chr8 E finished
chr9 Q finished
chr9 E finished
chr8 T finished
chr9 T finished
chr10 E finished
chr10 Q finished
chr11 Q finished
chr10 T finished
chr11 E finished
chr12 E finished
chr12 Q finished
chr11 T finished
chr13 E finished
chr13 Q finished
chr12 T finished
chr14 Q finished
chr13 T finished
chr14 E finished
chr15 Q finished
chr15 E finished
chr14 T finished
chr16 E finished
chr15 T finished
chr16 Q finished
chr17 Q finished
chr16 T finished
chr17 E finished
chr17 T finished
chr18 E finished
chr18 Q finished
chr19 Q finished
chr19 E finished
chr18 T finished
chr19 T finished
chr20 Q finished
chr20 E finished
chr21 Q finished
chr21 E

In [56]:
def chrom_iterator(input_dir, chrom_order, chrom_offset):
    for chrom in chrom_order:
        output_path = f'{input_dir}_{chrom}.npz'
        if not pathlib.Path(output_path).exists():
            continue
        chunk_size = 5000000
        data = load_npz(output_path).tocoo()
        df = pd.DataFrame({'bin1_id': data.row, 'bin2_id': data.col, 'count': data.data})
        df = df[df['bin1_id'] <= df['bin2_id']]
        for i, chunk_start in enumerate(range(0, df.shape[0], chunk_size)):
            chunk = df.iloc[chunk_start:chunk_start + chunk_size]
            chunk.iloc[:, :2] += chrom_offset[chrom]
            yield chunk


In [57]:
for matrix in ['Q', 'E', 'T']:
    output_path = f'{outdir}L2any_L2both_diffloop/{group_name}/{matrix}pv'
    cooler.create_cooler(cool_uri=f'{output_path}.cool',
                         bins=bins_df,
                         pixels=chrom_iterator(input_dir=output_path,
                                               chrom_order=chrom_sizes.index,
                                               chrom_offset=chrom_offset
                                              ),
                         ordered=True,
                         dtypes={'count': np.float32})


In [58]:
os.system(f'rm {outdir}L2any_L2both_diffloop/{group_name}/*pv_c*.npz')


0

In [59]:
leg = pd.Index(np.concatenate(ctgroup))
leg

Index(['c10-c0', 'c10-c4'], dtype='object')

In [60]:
loopall = [pd.read_csv(f'{coolpath[ct]}/{ct}.loop.bedpe', sep='\t', index_col=None, header=None) for ct in leg]
loopall = pd.concat(loopall, axis=0)
loopall = loopall.drop([6], axis=1).drop_duplicates(subset=[0,1,4]).sort_values([0,1,4])
loopall = pd.concat([loopall[(loopall[0]==c).values] for c in chrom_sizes.index])
loopall.index = np.arange(loopall.shape[0])
loopall


,0,1,2,3,4,5
0,chr1,960000,970000,chr1,1210000,1220000
1,chr1,1000000,1010000,chr1,1100000,1110000
2,chr1,1000000,1010000,chr1,1110000,1120000
3,chr1,1010000,1020000,chr1,1090000,1100000
4,chr1,1010000,1020000,chr1,1100000,1110000
...,...,...,...,...,...,...
479734,chr22,50570000,50580000,chr22,50670000,50680000
479735,chr22,50580000,50590000,chr22,50670000,50680000
479736,chr22,50590000,50600000,chr22,50670000,50680000
479737,chr22,50600000,50610000,chr22,50670000,50680000


In [61]:
loopall.to_csv(f'{outdir}L2any_L2both_diffloop/{group_name}/merged_loop.bedpe', sep='\t', index=False, header=False)
loopall.to_hdf(f'{outdir}L2any_L2both_diffloop/{group_name}/merged_loop.hdf', key='data')


In [62]:
for c in chrom_sizes.index:
    loopfilter = (loopall[0]==c)
    looptmp = loopall.loc[loopfilter, [1,4]].values // res
    for matrix in ['Q', 'E', 'T']:
        cool = cooler.Cooler(f'{outdir}L2any_L2both_diffloop/{group_name}/{matrix}pv.cool')
        pv = triu(cool.matrix(balance=False, sparse=True).fetch(c)).tocsr()
        loopall.loc[loopfilter, f'{matrix}anova'] = pv[(looptmp[:,0], looptmp[:,1])].A1
    print(c)


chr1
chr2
chr3
chr4
chr5
chr6
chr7
chr8
chr9
chr10
chr11
chr12
chr13
chr14
chr15
chr16
chr17
chr18
chr19
chr20
chr21
chr22


In [74]:
from scipy.stats import f
from statsmodels.sandbox.stats.multicomp import multipletests as FDR
    
def stats2fdr(data):
    pv = [f.sf(x, len(leg)-1, ncell-len(leg)) for x in data.values]
    fdr = FDR(pv, 0.01, "fdr_bh")[1]
    return fdr


In [75]:
cpu = 3
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for m in ['Q', 'E', 'T']:
        future = executor.submit(
            stats2fdr,
            data=loopall[f'{m}anova'],
        )
        futures[future] = m

    for future in as_completed(futures):
        m = futures[future]
        tmp = future.result()
        loopall[f'{m}fdr'] = tmp.copy()
    

In [77]:
loopall.to_hdf(f'{outdir}L2any_L2both_diffloop/{group_name}/merged_loop.hdf', key='data')


In [78]:
# get imputed q value at each contact from cool file 
def load_Q(ct, m):
    tmp = []
    cool_file = cooler.Cooler(f'{coolpath[ct]}/{ct}.{matrix}.cool').matrix(balance=False, sparse=True)
    for c in chrom_sizes.index:
        mat = cool_file.fetch(c).tocsr()
        # select for loopall contact q value (as 1d array)
        sel = (loopall[0]==c)
        if sel.sum()>0:
            tmp.append(mat[(loopall.loc[sel, 1].values // res, loopall.loc[sel, 4].values // res)].A1)
        # print(ct, c)
    return np.concatenate(tmp)


In [83]:
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for xx in leg:
        future = executor.submit(
            load_Q,
            ct=xx,
            m='Q'
        )
        futures[future] = xx

    loopq = []
    for future in as_completed(futures):
        tmp = future.result()
        ct = futures[future]
        loopq.append(pd.DataFrame(tmp, columns=[ct]))
        print(f'{ct} finished')
        

c10-c4 finished
c10-c0 finished


In [84]:
loopq = pd.concat(loopq, axis=1)
loopq = loopq[leg]


In [85]:
loopq.to_hdf(f'{outdir}L2any_L2both_diffloop/{group_name}/loop_Q.hdf', key='data')
